In [1]:
import torch
from diffusion.approaches.matching.prob_paths import (
    GaussianCondProbPath,
    LinearAlpha,
    LinearBeta,
)
from diffusion.approaches.matching.flow_trainer import FlowTrainer
from diffusion.sampleables.mnist_sampleable import MNISTSampleable
from diffusion.architectures.backbones.res_unet import ResUnet

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [3]:
sampeable = MNISTSampleable(train=True)
val_sampeable = MNISTSampleable(train=False)

path = GaussianCondProbPath(
    p_data=sampeable,
    p_simple_shape=sampeable.shape,
    alpha=LinearAlpha(),
    beta=LinearBeta(),
).to(device)

val_path = GaussianCondProbPath(
    p_data=val_sampeable,
    p_simple_shape=sampeable.shape,
    alpha=LinearAlpha(),
    beta=LinearBeta(),
).to(device)

backbone = ResUnet(
    in_channels=1,
    channels=[16, 32, 64],
    num_classes=sampeable.num_classes,
    t_dim=64,
    y_dim=32,
    cond_dim=64,
).to(device)

trainer = FlowTrainer(
    path=path,
    val_path=val_path,
    backbone=backbone,
    null_class=sampeable.num_classes,
)

In [4]:
trainer.train(
    num_epochs=15,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=500,
    validate=True,
)

2025-10-13 14:57:14,558 - flow-matching - INFO - Training model with size: 2.734 MiB
Epoch 0/15: 100%|██████████| 500/500 [00:38<00:00, 12.87it/s, train_loss=0.265527]
2025-10-13 14:57:53,759 - flow-matching - INFO - val loss: 0.17456752061843872
Epoch 1/15: 100%|██████████| 500/500 [00:38<00:00, 13.06it/s, train_loss=0.164505]
2025-10-13 14:58:32,276 - flow-matching - INFO - val loss: 0.15447764098644257
Epoch 2/15: 100%|██████████| 500/500 [00:38<00:00, 13.12it/s, train_loss=0.152111]
2025-10-13 14:59:10,639 - flow-matching - INFO - val loss: 0.14341726899147034
Epoch 3/15: 100%|██████████| 500/500 [00:38<00:00, 13.11it/s, train_loss=0.145188]
2025-10-13 14:59:49,023 - flow-matching - INFO - val loss: 0.13974781334400177
Epoch 4/15: 100%|██████████| 500/500 [00:38<00:00, 13.12it/s, train_loss=0.141310]
2025-10-13 15:00:27,369 - flow-matching - INFO - val loss: 0.13971154391765594
Epoch 5/15: 100%|██████████| 500/500 [00:37<00:00, 13.18it/s, train_loss=0.136186]
2025-10-13 15:01:05,54

OrderedDict([('start.weight',
              tensor([[[[-0.1788, -0.3266,  0.2598],
                        [ 0.0435, -0.3567,  0.1089],
                        [-0.2844, -0.2784,  0.1699]]],
              
              
                      [[[ 0.0041,  0.0416, -0.2293],
                        [-0.3353,  0.3818, -0.1643],
                        [-0.1359, -0.1778,  0.2173]]],
              
              
                      [[[ 0.2281,  0.3420,  0.2423],
                        [ 0.2634, -0.2762, -0.2256],
                        [ 0.2053,  0.0722,  0.0560]]],
              
              
                      [[[ 0.1799,  0.0298,  0.1915],
                        [-0.3526, -0.0964, -0.0357],
                        [ 0.0608, -0.0267, -0.2554]]],
              
              
                      [[[-0.3222,  0.2345,  0.0696],
                        [ 0.0275, -0.0070,  0.0806],
                        [ 0.0387,  0.1709, -0.2200]]],
              
              
               

In [5]:
torch.save(backbone.state_dict(), "./models/backbone_flow.pt")